## Ocupação Hospitalar

Ocupacao hospitalar 2020-2022 -> Silver. Compativel com Jupyter.
Cole TODO o arquivo numa celula e execute, ou use %run caminho/do/arquivo.py.
Dependencias: pandas, pyarrow. Nao altera Bronze nem acessa PostgreSQL.
Cada execucao cria uma pasta nova; nao ha retomada nesta versao.


In [4]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from collections import Counter
import hashlib
import json
import re
import sqlite3
import unicodedata
import uuid
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [5]:
PROJETO = Path(r'D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil')
BRONZE = PROJETO / 'Camada_Bronze/dados_ocupacao_hospitalar/parquet'
REFERENCIA_CNES = PROJETO / 'Camada_Silver/dados/cnes_leitos/20261027_153722_858fee3c/cnes_leitos_2020_2024.parquet'
LOTE = 50_000
ESPERADO = {2020: 554706, 2021: 745042, 2022: 291928}
# Limite operacional para REVISAO, nao limite clinico e nao exclusao automatica.
LIMIAR_REVISAO = 10_000
VERSAO = '1.0.0'
UF_NOMES = dict(zip(
    'AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO'.split(),
    ['Acre','Alagoas','Amapa','Amazonas','Bahia','Ceara','Distrito Federal',
     'Espirito Santo','Goias','Maranhao','Mato Grosso','Mato Grosso do Sul',
     'Minas Gerais','Para','Paraiba','Parana','Pernambuco','Piaui','Rio de Janeiro',
     'Rio Grande do Norte','Rio Grande do Sul','Rondonia','Roraima','Santa Catarina',
     'Sao Paulo','Sergipe','Tocantins']))
COLUNAS = ['indice_origem_csv','_id','dataNotificacao','cnes',
 'ocupacaoSuspeitoCli','ocupacaoSuspeitoUti','ocupacaoConfirmadoCli','ocupacaoConfirmadoUti',
 'ocupacaoCovidUti','ocupacaoCovidCli','ocupacaoHospitalarUti','ocupacaoHospitalarCli',
 'saidaSuspeitaObitos','saidaSuspeitaAltas','saidaConfirmadaObitos','saidaConfirmadaAltas',
 'origem','_p_usuario','estadoNotificacao','municipioNotificacao','estado','municipio',
 'excluido','validado','_created_at','_updated_at']
MEDIDAS = [c for c in COLUNAS if c.startswith(('ocupacao','saida'))]
DATAS = {'dataNotificacao':'notificacao_utc','_created_at':'criacao_utc','_updated_at':'atualizacao_utc'}

In [6]:
def normalizar(x):
    if x is None or pd.isna(x): return ''
    return re.sub('[^A-Z0-9]', '', unicodedata.normalize('NFKD', str(x)).encode('ascii','ignore').decode().upper())


def texto(s):
    return s.astype('string').str.strip().replace('', pd.NA)


def snake(x):
    return re.sub(r'(?<!^)(?=[A-Z])', '_', x).lower()


def hash_arquivo(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda: f.read(8*1024*1024), b''): h.update(b)
    return h.hexdigest()


def candidato_cnes(x):
    if x is None or pd.isna(x) or not str(x).strip(): return None, 'AUSENTE'
    s = str(x).strip()
    if re.fullmatch(r'[0-9]{7}', s):
        return (s, 'ORIGINAL_7_DIGITOS') if s != '0000000' else (None, 'CODIGO_ZERADO')
    m = re.fullmatch(r'([0-9]{1,7})(?:\.0+)?', s)
    if m:
        codigo = m[1].zfill(7)
        if codigo == '0000000': return None, 'CODIGO_ZERADO'
        return codigo, 'REMOVIDO_DECIMAL_ZERO_E_OU_COMPLETADO_ZEROS'
    m = re.fullmatch(r'([0-9]{7})\s+-\s+\S.*', s)
    if m and m[1] != '0000000': return m[1], 'CODIGO_INICIAL_SEPARADO_DO_NOME'
    return None, 'FORMATO_NAO_TRATADO'


def quantidade(x):
    if x is None or pd.isna(x) or not str(x).strip(): return None, 'AUSENTE', False
    try:
        n = Decimal(str(x).strip())
        if not n.is_finite(): return None, 'NAO_FINITO', False
        if n < 0: return None, 'NEGATIVO', False
        if n != n.to_integral_value(): return None, 'FRACIONARIO', False
        if n > 9223372036854775807: return None, 'FORA_BIGINT', True
        v = int(n)
        return v, 'INTEIRO_NAO_NEGATIVO', v > LIMIAR_REVISAO
    except (InvalidOperation, ValueError): return None, 'NAO_NUMERICO', False


def parse_datas(s):
    v = texto(s)
    # Nao atribui UTC a timestamps sem fuso explicito.
    formato = v.str.fullmatch(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?(?:Z|[+-]\d{2}:\d{2})', na=False)
    d = pd.to_datetime(v.where(formato), format='ISO8601', errors='coerce', utc=True)
    st = pd.Series('VALIDA', index=s.index, dtype='string')
    st.loc[d.isna()] = 'INVALIDA_OU_FORMATO_NAO_PREVISTO'
    st.loc[v.isna()] = 'AUSENTE'
    return d, st


def carregar_referencia():
    r = pd.read_parquet(REFERENCIA_CNES)
    necessarias = {'cnes','competencia','uf','municipio','municipio_ibge','codigo_ibge_7'}
    if not necessarias <= set(r): raise ValueError(f'Faltam colunas CNES: {necessarias-set(r)}')
    for c in necessarias: r[c] = texto(r[c])
    if r.duplicated(['cnes','competencia']).any(): raise ValueError('Referencia CNES com chave repetida.')
    if not r['cnes'].str.fullmatch(r'[0-9]{7}', na=False).all(): raise ValueError('CNES da referencia fora do formato.')
    if not r['competencia'].str.fullmatch(r'20[0-9]{2}(0[1-9]|1[0-2])', na=False).all(): raise ValueError('Competencia CNES invalida.')
    r['_uf'] = r['uf'].map(normalizar)
    r['_mun1'] = r['municipio'].map(normalizar)
    r['_mun2'] = r['municipio_ibge'].map(normalizar)
    return r


def criar_esquema():
    f = [pa.field(snake(c), pa.string()) for c in COLUNAS]
    for c in ['arquivo_origem','tratado_em_utc','cnes_candidato','metodo_cnes','cnes_vinculado',
              'competencia_referencia_utc','uf_padronizada','status_vinculo_cnes',
              'codigo_ibge_7','municipio_ibge','uf_ibge','perfil_campos_ocupacao']:
        f.append(pa.field(c, pa.string()))
    for c in ['ano_arquivo','linha_arquivo','ano_referencia_utc']: f.append(pa.field(c, pa.int64()))
    f.append(pa.field('dia_notificacao_utc', pa.date32()))
    for c in DATAS.values():
        f.extend([pa.field(c, pa.timestamp('us',tz='UTC')), pa.field('status_'+c, pa.string())])
    for c in ['excluido','validado']:
        f.extend([pa.field(c+'_booleano',pa.bool_()), pa.field('status_'+c,pa.string())])
    for c in MEDIDAS:
        n = snake(c)
        f.extend([pa.field(n+'_quantidade',pa.int64()),pa.field('status_'+n,pa.string()),pa.field('fl_extremo_'+n,pa.bool_())])
    for c in ['fl_quantidade_invalida','fl_valor_extremo','fl_localizacao_notificacao_divergente',
              'fl_ano_utc_diferente_arquivo','fl_atualizacao_antes_criacao']:
        f.append(pa.field(c,pa.bool_()))
    return pa.schema(f)


def tratar(df, ano, inicio, ref, contagens, instante):
    out = df.copy(); out.columns = [snake(c) for c in out]
    out['arquivo_origem'] = f'ocupacao_{ano}.parquet'
    out['ano_arquivo'] = ano; out['linha_arquivo'] = range(inicio+1,inicio+len(df)+1)
    out['tratado_em_utc'] = instante
    for origem, destino in DATAS.items():
        out[destino], out['status_'+destino] = parse_datas(df[origem])
        contagens.update({'datas/'+destino+'/'+k:int(v) for k,v in out['status_'+destino].value_counts().items()})
    dt = out['notificacao_utc']
    out['dia_notificacao_utc'] = dt.dt.date
    out['ano_referencia_utc'] = dt.dt.year.astype('Int64')
    out['competencia_referencia_utc'] = dt.dt.strftime('%Y%m').astype('string')
    out['fl_ano_utc_diferente_arquivo'] = (dt.notna() & dt.dt.year.ne(ano)).fillna(False)
    out['fl_atualizacao_antes_criacao'] = (out['atualizacao_utc'] < out['criacao_utc']).fillna(False)
    for c in ['excluido','validado']:
        v = texto(df[c]).str.lower()
        out[c+'_booleano'] = v.map({'true':True,'false':False}).astype('boolean')
        out['status_'+c] = pd.Series('VALIDO',index=df.index,dtype='string').mask(out[c+'_booleano'].isna(),'INVALIDO').mask(v.isna(),'AUSENTE')
        contagens.update({'flags/'+c+'/'+str(k):int(q) for k,q in v.fillna('<AUSENTE>').value_counts().items()})
    out['fl_quantidade_invalida'] = False; out['fl_valor_extremo'] = False
    for c in MEDIDAS:
        cache = {x:quantidade(x) for x in df[c].dropna().unique()}
        resultados = [cache.get(x,(None,'AUSENTE',False)) if pd.notna(x) else (None,'AUSENTE',False) for x in df[c]]
        n = snake(c)
        out[n+'_quantidade'] = pd.array([r[0] for r in resultados],dtype='Int64')
        out['status_'+n] = [r[1] for r in resultados]
        out['fl_extremo_'+n] = [r[2] for r in resultados]
        out['fl_quantidade_invalida'] |= ~out['status_'+n].isin(['AUSENTE','INTEIRO_NAO_NEGATIVO'])
        out['fl_valor_extremo'] |= out['fl_extremo_'+n]
        contagens.update({'quantidades/'+c+'/'+str(k):int(q) for k,q in out['status_'+n].value_counts().items()})
    antigos = df[[c for c in MEDIDAS if c.startswith(('ocupacaoSuspeito','ocupacaoConfirmado'))]].apply(texto).notna().any(axis=1)
    novos = df[[c for c in MEDIDAS if c.startswith(('ocupacaoCovid','ocupacaoHospitalar'))]].apply(texto).notna().any(axis=1)
    perfil = pd.Series('NENHUM',index=df.index)
    perfil.loc[antigos] = 'SUSPEITO_CONFIRMADO'
    perfil.loc[novos] = 'COVID_HOSPITALAR'
    perfil.loc[antigos & novos] = 'AMBOS'
    out['perfil_campos_ocupacao'] = perfil
    candidatos = [candidato_cnes(x) for x in df['cnes']]
    out['cnes_candidato'] = [x[0] for x in candidatos]; out['metodo_cnes'] = [x[1] for x in candidatos]
    mapa_uf = {normalizar(nome):uf for uf,nome in UF_NOMES.items()} | {u:u for u in UF_NOMES}
    out['uf_padronizada'] = df['estado'].map(normalizar).map(mapa_uf)
    municipio = df['municipio'].map(normalizar)
    out['fl_localizacao_notificacao_divergente'] = (
        df['estadoNotificacao'].map(normalizar).map(mapa_uf).fillna('') != out['uf_padronizada'].fillna('')
    ) | (df['municipioNotificacao'].map(normalizar) != municipio)
    # Merge many-to-one assegura que nenhuma linha seja multiplicada.
    chaves = pd.DataFrame({'cnes':out['cnes_candidato'],'competencia':out['competencia_referencia_utc']})
    link = chaves.merge(ref,on=['cnes','competencia'],how='left',validate='many_to_one',sort=False)
    achou = link['_uf'].notna()
    local_ok = out['uf_padronizada'].eq(link['_uf']).fillna(False) & municipio.ne('') & (municipio.eq(link['_mun1']) | municipio.eq(link['_mun2']))
    st = pd.Series('SEM_CORRESPONDENCIA_NA_REFERENCIA_MENSAL',index=df.index,dtype='string')
    st.loc[achou] = 'LOCALIZACAO_DIVERGENTE_OU_AUSENTE'
    st.loc[achou & local_ok] = 'VINCULADO_CNES_COMPETENCIA_LOCALIZACAO'
    st.loc[out['cnes_candidato'].isna()] = 'CNES_SEM_CANDIDATO'
    st.loc[dt.isna()] = 'DATA_NOTIFICACAO_INVALIDA'
    ok = st.eq('VINCULADO_CNES_COMPETENCIA_LOCALIZACAO')
    out['status_vinculo_cnes'] = st
    out['cnes_vinculado'] = out['cnes_candidato'].where(ok)
    out['codigo_ibge_7'] = link['codigo_ibge_7'].where(ok)
    out['municipio_ibge'] = link['municipio_ibge'].where(ok)
    out['uf_ibge'] = link['uf'].where(ok)
    for nome in ['status_vinculo_cnes','metodo_cnes','perfil_campos_ocupacao']:
        contagens.update({nome+'/'+str(k):int(q) for k,q in out[nome].value_counts().items()})
    for nome in [c for c in out if c.startswith('fl_')]: contagens['avisos/'+nome] += int(out[nome].sum())
    return out


def ddl(schema):
    def tipo(t):
        if pa.types.is_string(t): return 'TEXT'
        if pa.types.is_boolean(t): return 'BOOLEAN'
        if pa.types.is_timestamp(t): return 'TIMESTAMPTZ'
        if pa.types.is_date(t): return 'DATE'
        return 'BIGINT'
    campos = [f'    "{f.name}" {tipo(f.type)}' for f in schema]
    campos.append('    PRIMARY KEY ("_id")')
    return '-- Executar somente apos revisao do relatorio.\nCREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE silver.ocupacao_hospitalar (\n'+',\n'.join(campos)+'\n);\n'


def main():
    print('Iniciando tratamento da ocupacao hospitalar...',flush=True)
    for ano in ESPERADO:
        p = BRONZE/f'ocupacao_{ano}.parquet'
        if not p.is_file(): raise FileNotFoundError(p)
        with pq.ParquetFile(p) as pf:
            if pf.schema_arrow.names != COLUNAS: raise ValueError(f'Layout diferente: {p}')
            if pf.metadata.num_rows != ESPERADO[ano]: raise ValueError(f'Contagem diferente: {p}')
    if not REFERENCIA_CNES.is_file(): raise FileNotFoundError(REFERENCIA_CNES)
    ref = carregar_referencia()
    run = datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    base = PROJETO/'Camada_Silver'
    dados = base/'dados/ocupacao_hospitalar'/run
    qualidade = base/'qualidade/ocupacao_hospitalar'/run
    doc = base/'documentacao/ocupacao_hospitalar'/run
    for p in [dados,qualidade,doc]: p.mkdir(parents=True,exist_ok=False)
    schema = criar_esquema(); contagens = Counter(); pendencias = Counter()
    rel = {'status':'EM_EXECUCAO','versao':VERSAO,'registros':0,'anos_arquivo':[2020,2021,2022],
           'anos_nao_localizados_na_coleta':[2023,2024], 'registros_por_ano_arquivo':{},
           'referencia_cnes':str(REFERENCIA_CNES),'sha256_referencia':hash_arquivo(REFERENCIA_CNES),
           'limiar_revisao_operacional':LIMIAR_REVISAO,'entradas':[]}
    destino = dados/'ocupacao_2020_2022.parquet'; parcial = destino.with_suffix('.parquet.part')
    banco = sqlite3.connect(qualidade/'auditoria.sqlite')
    banco.execute('CREATE TABLE ids (id TEXT PRIMARY KEY)')
    banco.execute('CREATE TABLE dias (cnes TEXT, dia TEXT, n INTEGER, ativos INTEGER, PRIMARY KEY(cnes,dia))')
    instante = datetime.now(timezone.utc).isoformat()
    try:
        with pq.ParquetWriter(parcial,schema,compression='snappy') as writer:
            for ano, esperado in ESPERADO.items():
                arquivo = BRONZE/f'ocupacao_{ano}.parquet'; h = hash_arquivo(arquivo); processados = 0
                print(f'Tratando {arquivo.name}',flush=True)
                with pq.ParquetFile(arquivo) as pf:
                    for batch in pf.iter_batches(batch_size=LOTE):
                        df = batch.to_pandas().reset_index(drop=True)
                        ids = texto(df['_id'])
                        if ids.isna().any(): raise ValueError('ID ausente: carga bloqueada.')
                        # Violacao da PK interrompe: nenhuma deduplicacao silenciosa.
                        banco.executemany('INSERT INTO ids VALUES (?)',[(x,) for x in ids])
                        out = tratar(df,ano,processados,ref,contagens,instante)
                        writer.write_table(pa.Table.from_pandas(out,schema=schema,preserve_index=False,safe=True))
                        nao_vinculados = out['status_vinculo_cnes'].ne('VINCULADO_CNES_COMPETENCIA_LOCALIZACAO')
                        cols = ['cnes','cnes_candidato','metodo_cnes','estado','municipio','status_vinculo_cnes']
                        for chave,q in out.loc[nao_vinculados,cols].fillna('<AUSENTE>').value_counts().items():
                            pendencias[(ano,)+tuple(chave)] += int(q)
                        validos = out['cnes_vinculado'].notna() & out['dia_notificacao_utc'].notna()
                        dia = out.loc[validos,['cnes_vinculado','dia_notificacao_utc','excluido_booleano']].copy()
                        dia['ativo'] = dia['excluido_booleano'].eq(False).fillna(False).astype(int)
                        grupos = dia.groupby(['cnes_vinculado','dia_notificacao_utc']).agg(n=('ativo','size'),ativos=('ativo','sum')).reset_index()
                        banco.executemany('INSERT INTO dias VALUES (?,?,?,?) ON CONFLICT(cnes,dia) DO UPDATE SET n=n+excluded.n, ativos=ativos+excluded.ativos',
                            [(r[0],str(r[1]),int(r[2]),int(r[3])) for r in grupos.itertuples(index=False,name=None)])
                        banco.commit(); processados += len(df); rel['registros'] += len(df)
                        print(f'  Processados: {processados:,}/{esperado:,}',flush=True)
                if processados != esperado: raise ValueError('Contagem final divergente.')
                if hash_arquivo(arquivo) != h: raise ValueError('Entrada alterada durante leitura.')
                rel['registros_por_ano_arquivo'][str(ano)] = processados
                rel['entradas'].append({'arquivo':str(arquivo),'sha256':h})
        if hash_arquivo(REFERENCIA_CNES) != rel['sha256_referencia']: raise ValueError('Referencia alterada durante execucao.')
        with pq.ParquetFile(parcial) as pf:
            if pf.metadata.num_rows != sum(ESPERADO.values()): raise ValueError('Contagem Parquet incorreta.')
        rel['ids_distintos'] = banco.execute('SELECT COUNT(*) FROM ids').fetchone()[0]
        rel['grupos_cnes_dia_utc_repetidos_vinculados'] = banco.execute('SELECT COUNT(*) FROM dias WHERE n>1').fetchone()[0]
        rel['grupos_cnes_dia_utc_com_multiplos_nao_excluidos'] = banco.execute('SELECT COUNT(*) FROM dias WHERE ativos>1').fetchone()[0]
        pd.read_sql_query('SELECT * FROM dias WHERE n>1 ORDER BY n DESC',banco).to_csv(qualidade/'repeticoes_cnes_dia_utc.csv',sep=';',index=False,encoding='utf-8-sig')
        linhas = [dict(zip(['ano_arquivo','cnes_original','cnes_candidato','metodo_cnes','estado','municipio','status'],k),registros=v) for k,v in pendencias.items()]
        pd.DataFrame(linhas,columns=['ano_arquivo','cnes_original','cnes_candidato','metodo_cnes','estado','municipio','status','registros']).to_csv(qualidade/'pendencias_cnes.csv',sep=';',index=False,encoding='utf-8-sig')
        (doc/'criar_tabela_ocupacao_hospitalar.sql').write_text(ddl(schema),encoding='utf-8')
        regras = '''Originais preservados como texto; nenhuma linha removida. _id unico nesta coleta, nao identifica pessoa.
Datas UTC com fuso explicito; dia/competencia UTC sao referencias tecnicas, nao dia local validado.
Referencia CNES mensal pelo mes UTC; match exige UF e municipio normalizados compativeis.
Normalizacao de municipio remove acentos, espacos e pontuacao; sem fuzzy match ou equivalencias manuais.
CNES candidato separado do vinculado. Codigo longo nao truncado; sem match nao prova cadastro inexistente.
Municipio IBGE herdado somente do CNES vinculado. Nao ha fallback automatico por nome.
Quantidades negativas/fracionarias/nao finitas ficam nulas nas derivadas; originais e status preservados.
Inteiros nao negativos nao sao garantia de plausibilidade; acima do limiar operacional ha flag de revisao.
Nenhuma soma ou coalesce entre campos de ocupacao suspeita/confirmada/COVID/hospitalar.
Ausente nao equivale a zero. Excluido preservado; validado=False nao usado como filtro automatico.
Nao calcular taxas de ocupacao com denominadores CNES sem antes confirmar compatibilidade conceitual.
Repeticoes hospital/dia UTC sao reportadas somente para CNES vinculado, sem escolher ultima notificacao.
Contagens por campo/flag podem se sobrepor. Campos de saida nao sao somados com estoques de ocupacao.
2023/2024 nao localizados nesta coleta; isso nao representa ocupacao zero ou ausencia de fonte oficial.
'''
        (doc/'regras_tratamento.txt').write_text(regras,encoding='utf-8')
        parcial.replace(destino)
        rel.update(status='REVISAR_ANTES_DA_CARGA',parquet_silver=str(destino),sha256_parquet=hash_arquivo(destino))
    except Exception as e:
        rel.update(status='INTERROMPIDO_NAO_CARREGAR',erro=str(e))
        raise
    finally:
        banco.close(); rel['contagens'] = dict(sorted(contagens.items()))
        (qualidade/'relatorio_qualidade.json').write_text(json.dumps(rel,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f"\nStatus: {rel['status']}\nRegistros: {rel['registros']:,}\nDados: {destino}\nQualidade: {qualidade}\nSQL: {doc/'criar_tabela_ocupacao_hospitalar.sql'}",flush=True)
    print('Vinculos:',{k:v for k,v in contagens.items() if k.startswith('status_vinculo_cnes/')})
    print('Nenhuma carga realizada no PostgreSQL.')
    return rel


if __name__ == '__main__':
    resultado_ocupacao = main()

Iniciando tratamento da ocupacao hospitalar...
Tratando ocupacao_2020.parquet
  Processados: 50,000/554,706
  Processados: 100,000/554,706
  Processados: 150,000/554,706
  Processados: 200,000/554,706
  Processados: 250,000/554,706
  Processados: 300,000/554,706
  Processados: 350,000/554,706
  Processados: 400,000/554,706
  Processados: 450,000/554,706
  Processados: 500,000/554,706
  Processados: 550,000/554,706
  Processados: 554,706/554,706
Tratando ocupacao_2021.parquet
  Processados: 50,000/745,042
  Processados: 100,000/745,042
  Processados: 150,000/745,042
  Processados: 200,000/745,042
  Processados: 250,000/745,042
  Processados: 300,000/745,042
  Processados: 350,000/745,042
  Processados: 400,000/745,042
  Processados: 450,000/745,042
  Processados: 500,000/745,042
  Processados: 550,000/745,042
  Processados: 600,000/745,042
  Processados: 650,000/745,042
  Processados: 700,000/745,042
  Processados: 745,042/745,042
Tratando ocupacao_2022.parquet
  Processados: 50,000/291

In [7]:
from pathlib import Path
import json
import pandas as pd

PASTA = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\qualidade\ocupacao_hospitalar"
    r"\20261029_204406_8732ff74"
)

relatorio = json.loads(
    (PASTA / "relatorio_qualidade.json").read_text(encoding="utf-8")
)

print("RELATÓRIO DE QUALIDADE")
print(json.dumps(relatorio, ensure_ascii=False, indent=2))

pendencias = pd.read_csv(
    PASTA / "pendencias_cnes.csv",
    sep=";",
    dtype="string",
    keep_default_na=False,
)
pendencias["registros"] = pd.to_numeric(
    pendencias["registros"], errors="raise"
)

# Agrupa os anos para destacar problemas recorrentes.
resumo = (
    pendencias.groupby(
        [
            "cnes_original",
            "cnes_candidato",
            "metodo_cnes",
            "estado",
            "municipio",
            "status",
        ],
        dropna=False,
        as_index=False,
    )["registros"]
    .sum()
    .sort_values("registros", ascending=False)
)

for status, grupo in resumo.groupby("status", sort=False):
    print(f"\n{status} — 15 COMBINAÇÕES MAIS FREQUENTES")
    print(grupo.head(15).to_string(index=False))

RELATÓRIO DE QUALIDADE
{
  "status": "REVISAR_ANTES_DA_CARGA",
  "versao": "1.0.0",
  "registros": 1591676,
  "anos_arquivo": [
    2020,
    2021,
    2022
  ],
  "anos_nao_localizados_na_coleta": [
    2023,
    2024
  ],
  "registros_por_ano_arquivo": {
    "2020": 554706,
    "2021": 745042,
    "2022": 291928
  },
  "referencia_cnes": "D:\\Cleber\\GitHub\\Projetos\\observatorio_covid19_brasil\\Camada_Silver\\dados\\cnes_leitos\\20261027_153722_858fee3c\\cnes_leitos_2020_2024.parquet",
  "sha256_referencia": "467684272ff16984a8dbddec35b747e09f96f2e6faf480e01c2fab058348fbe0",
  "limiar_revisao_operacional": 10000,
  "entradas": [
    {
      "arquivo": "D:\\Cleber\\GitHub\\Projetos\\observatorio_covid19_brasil\\Camada_Bronze\\dados_ocupacao_hospitalar\\parquet\\ocupacao_2020.parquet",
      "sha256": "485ba9048b33f4554db0623dd2dc6a2ddae7b9baf13dc0408f8ea8940aa0d25b"
    },
    {
      "arquivo": "D:\\Cleber\\GitHub\\Projetos\\observatorio_covid19_brasil\\Camada_Bronze\\dados_ocupaca

In [8]:
from pathlib import Path
import pandas as pd

BASE = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver"
)

ARQUIVO_OCUPACAO = (
    BASE / "dados/ocupacao_hospitalar/20261029_204406_8732ff74"
    / "ocupacao_2020_2022.parquet"
)

ARQUIVO_CNES = (
    BASE / "dados/cnes_leitos/20261027_153722_858fee3c"
    / "cnes_leitos_2020_2024.parquet"
)

ocupacao = pd.read_parquet(
    ARQUIVO_OCUPACAO,
    columns=[
        "cnes_candidato",
        "competencia_referencia_utc",
        "estado",
        "municipio",
        "status_vinculo_cnes",
    ],
)

divergentes = ocupacao.loc[
    ocupacao["status_vinculo_cnes"].eq(
        "LOCALIZACAO_DIVERGENTE_OU_AUSENTE"
    )
].copy()

del ocupacao

cnes = pd.read_parquet(
    ARQUIVO_CNES,
    columns=[
        "cnes", "competencia", "uf", "municipio",
        "municipio_ibge", "nome_estabelecimento",
    ],
).rename(columns={
    "municipio": "municipio_cnes",
    "uf": "uf_cnes",
})

for coluna in ["cnes", "competencia"]:
    cnes[coluna] = cnes[coluna].astype("string").str.strip()

comparacao = divergentes.merge(
    cnes,
    left_on=["cnes_candidato", "competencia_referencia_utc"],
    right_on=["cnes", "competencia"],
    how="left",
    validate="many_to_one",
)

resumo = (
    comparacao.groupby(
        [
            "cnes_candidato",
            "estado",
            "municipio",
            "uf_cnes",
            "municipio_cnes",
            "municipio_ibge",
            "nome_estabelecimento",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

print("LOCALIZAÇÃO INFORMADA × REFERÊNCIA CNES")
print(resumo.head(30).to_string(index=False))
print(f"\nRegistros comparados: {len(comparacao):,}")

LOCALIZAÇÃO INFORMADA × REFERÊNCIA CNES
cnes_candidato            estado             municipio uf_cnes           municipio_cnes           municipio_ibge                                  nome_estabelecimento  registros
       2302969    Santa Catarina         Florianópolis      SC                 SAO JOSE                 São José                                                  ICSC        691
       0013633            Paraná               Colombo      PR    CAMPINA GRANDE DO SUL    Campina Grande do Sul                               HOSPITAL ANGELINA CARON        623
       0011800    Espírito Santo            Vila Velha      ES                  VITORIA                  Vitória             HOSPITAL INFANTIL NOSSA SENHORA DA GLORIA        533
       6945368    Espírito Santo              Anchieta      ES               ITAPEMIRIM               Itapemirim                                       HECI ITAPEMIRIM        471
       2665883    Santa Catarina               Tubarão      SC        

## PostgreSQL

In [9]:
from pathlib import Path
from getpass import getpass
import hashlib
import json

import pyarrow.parquet as pq
import psycopg
from psycopg import sql


BASE = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver"
)

EXECUCAO = "20261029_204406_8732ff74"

ARQUIVO = (
    BASE / "dados/ocupacao_hospitalar" / EXECUCAO
    / "ocupacao_2020_2022.parquet"
)

RELATORIO = (
    BASE / "qualidade/ocupacao_hospitalar" / EXECUCAO
    / "relatorio_qualidade.json"
)

CONEXAO = {
    "host": "localhost",
    "port": 5432,
    "dbname": "observatorio_covid19_brasil",
    "user": "postgres",
}

LOTE = 5_000


def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def carregar_ocupacao():
    if not ARQUIVO.is_file():
        raise FileNotFoundError(ARQUIVO)

    rel = json.loads(RELATORIO.read_text(encoding="utf-8"))

    if rel["status"] != "REVISAR_ANTES_DA_CARGA":
        raise RuntimeError("Status diferente do relatório revisado.")

    esperado = int(rel["registros"])
    if esperado != 1_591_676:
        raise RuntimeError("Total diferente da execução revisada.")

    print("Conferindo integridade do Parquet...", flush=True)

    if sha256_arquivo(ARQUIVO) != rel["sha256_parquet"]:
        raise RuntimeError("O Parquet foi alterado após o tratamento.")

    senha = getpass("Senha do PostgreSQL: ")

    with pq.ParquetFile(ARQUIVO) as parquet:
        if parquet.metadata.num_rows != esperado:
            raise RuntimeError("Contagem do Parquet diverge do relatório.")

        colunas = parquet.schema_arrow.names

        with psycopg.connect(**CONEXAO, password=senha) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT to_regclass('silver.ocupacao_hospitalar')"
                )
                if cur.fetchone()[0] is None:
                    raise RuntimeError(
                        "A tabela silver.ocupacao_hospitalar não existe."
                    )

                # Evita alterações simultâneas durante a carga.
                cur.execute("""
                    LOCK TABLE silver.ocupacao_hospitalar
                    IN EXCLUSIVE MODE
                """)

                cur.execute("""
                    SELECT EXISTS (
                        SELECT 1 FROM silver.ocupacao_hospitalar
                    )
                """)
                if cur.fetchone()[0]:
                    raise RuntimeError(
                        "A tabela já contém dados. "
                        "Carga interrompida para evitar duplicação."
                    )

                cur.execute("""
                    SELECT column_name
                    FROM information_schema.columns
                    WHERE table_schema = 'silver'
                      AND table_name = 'ocupacao_hospitalar'
                    ORDER BY ordinal_position
                """)
                colunas_banco = [linha[0] for linha in cur.fetchall()]

                if colunas_banco != colunas:
                    raise RuntimeError(
                        "Colunas da tabela e do Parquet divergem."
                    )

                comando = sql.SQL(
                    "COPY silver.ocupacao_hospitalar ({}) FROM STDIN"
                ).format(
                    sql.SQL(", ").join(map(sql.Identifier, colunas))
                )

                enviados = 0
                proximo_aviso = 100_000

                print(f"Carregando {esperado:,} registros...", flush=True)

                with cur.copy(comando) as copia:
                    for lote in parquet.iter_batches(batch_size=LOTE):
                        valores = [
                            coluna.to_pylist()
                            for coluna in lote.columns
                        ]

                        for linha in zip(*valores):
                            copia.write_row(linha)

                        enviados += lote.num_rows

                        if enviados >= proximo_aviso or enviados == esperado:
                            print(
                                f"Enviados: {enviados:,} / {esperado:,}",
                                flush=True,
                            )
                            proximo_aviso += 100_000

                if enviados != esperado:
                    raise RuntimeError("Total enviado divergente.")

                print("Conferindo a carga...", flush=True)

                cur.execute("""
                    SELECT ano_arquivo, COUNT(*)
                    FROM silver.ocupacao_hospitalar
                    GROUP BY ano_arquivo
                """)
                anuais = dict(cur.fetchall())

                anuais_esperados = {
                    int(ano): int(q)
                    for ano, q in rel["registros_por_ano_arquivo"].items()
                }
                if anuais != anuais_esperados:
                    raise RuntimeError(f"Contagens anuais divergentes: {anuais}")

                cur.execute("""
                    SELECT status_vinculo_cnes, COUNT(*)
                    FROM silver.ocupacao_hospitalar
                    GROUP BY status_vinculo_cnes
                """)
                vinculos = dict(cur.fetchall())

                prefixo = "status_vinculo_cnes/"
                vinculos_esperados = {
                    chave.removeprefix(prefixo): int(valor)
                    for chave, valor in rel["contagens"].items()
                    if chave.startswith(prefixo)
                }
                if vinculos != vinculos_esperados:
                    raise RuntimeError("Contagens de vínculos divergentes.")

                cur.execute("""
                    SELECT
                        COUNT(*) FILTER (WHERE excluido_booleano IS TRUE),
                        COUNT(*) FILTER (WHERE validado_booleano IS FALSE),
                        COUNT(*) FILTER (WHERE fl_quantidade_invalida IS TRUE),
                        COUNT(*) FILTER (WHERE fl_valor_extremo IS TRUE),
                        COUNT(*) FILTER (
                            WHERE notificacao_utc IS NULL
                               OR criacao_utc IS NULL
                               OR atualizacao_utc IS NULL
                        ),
                        COUNT(*) FILTER (WHERE cnes_vinculado IS NULL),
                        COUNT(*) FILTER (
                            WHERE perfil_campos_ocupacao = 'AMBOS'
                        )
                    FROM silver.ocupacao_hospitalar
                """)

                ressalvas = cur.fetchone()
                esperado_ressalvas = (
                    507,        # Marcados como excluídos
                    1_591_676,  # validado=False na fonte
                    1_726,      # Alguma quantidade inválida
                    15,         # Algum valor extremo sinalizado
                    0,          # Datas derivadas ausentes
                    80_508,     # Sem vínculo confirmado com CNES
                    6_833,      # Ambos os conjuntos de ocupação preenchidos
                )

                if ressalvas != esperado_ressalvas:
                    raise RuntimeError(
                        f"Ressalvas divergentes: {ressalvas}"
                    )

            # O contexto da conexão confirma a transação ao sair sem erro.

    print("\nCarga concluída e confirmada!")
    print(f"Registros no banco: {sum(anuais.values()):,}")
    print("Contagens anuais, vínculos e principais ressalvas conferidos.")
    print("Registros excluídos na fonte e pendências preservados.")


carregar_ocupacao()

Conferindo integridade do Parquet...
Carregando 1,591,676 registros...
Enviados: 100,000 / 1,591,676
Enviados: 200,000 / 1,591,676
Enviados: 300,000 / 1,591,676
Enviados: 400,000 / 1,591,676
Enviados: 500,000 / 1,591,676
Enviados: 600,000 / 1,591,676
Enviados: 700,000 / 1,591,676
Enviados: 800,000 / 1,591,676
Enviados: 900,000 / 1,591,676
Enviados: 1,000,000 / 1,591,676
Enviados: 1,100,000 / 1,591,676
Enviados: 1,200,000 / 1,591,676
Enviados: 1,300,000 / 1,591,676
Enviados: 1,400,000 / 1,591,676
Enviados: 1,500,000 / 1,591,676
Enviados: 1,591,676 / 1,591,676
Conferindo a carga...

Carga concluída e confirmada!
Registros no banco: 1,591,676
Contagens anuais, vínculos e principais ressalvas conferidos.
Registros excluídos na fonte e pendências preservados.
